# Unified Gradient Importance - Qwen3-1.7B (parameterized: gene-only / full-input)

Single notebook that replaces the `[Gradient Importance]` and `[NEW_Gradient_Importance]...GENE_ONLY...` variants. The `GENE_ONLY` switch, model id, data filename pattern, and gradient/plot knobs are **parameters** set by the Colab frontend (`colab_frontends/run_explain_colab.ipynb`) via papermill. Default (`GENE_ONLY=False`) runs on the committed `*_sorted.json` files.

In [ ]:
# ===== Papermill parameters =====
# Default = full-input "[Gradient Importance] 1K cleaned" variant (runs on the committed
# *_sorted.json files). GENE_ONLY=True selects the gene-only model + gene-only data files,
# which must be supplied (they are not committed to this repo).
GENE_ONLY = False

# Model / data ("" -> derived from GENE_ONLY in the next cell)
MODEL_ID = ""
FILENAME_PATTERN = ""
REPO_URL = "https://github.com/HangYu8123/SC_Ageing_Prediction.git"
PROJECT_DIR = "/content/SC_Ageing_Prediction"
DATA_DIR = ""                  # "" -> {PROJECT_DIR}/fine_tune_chunks (or $SCAP_DATA_DIR)

# Tokenizer / model
MAX_LENGTH = 1024
DTYPE = "bfloat16"             # "bfloat16" | "float16" | "float32"

# Gradient analysis controls
GRAD_BATCH_SIZE = 8
GRAD_TOPK = 100
SAMPLES_PER_ORGAN = None       # int for a fast preview; None = all samples
USE_TRUE_LABEL = True

# Dataset scope
ORGAN_KEYS = ["bladder", "brain", "bone", "limb", "kidney", "liver", "lung", "heart"]
ALL_PARTS = list(range(1, 12))
SEED = 42

# Output / persistence
SAVE_RESULTS = False
OUTPUT_DIR = ""               # "" -> {PROJECT_DIR}/explain_outputs
MOUNT_DRIVE = False

# Plot controls
PLOT_TOP_N = 100
PLOT_OUT_DIR = None           # None -> inline only; set a dir to also write PNGs
PLOT_BAR_COLOR = "steelblue"
PLOT_WORDCLOUD_CMAP = "plasma"
PLOT_FIGSIZE = (16, 20)
RELEASE_RUNTIME = False


## 1. Resolve parameters and Colab guards

In [ ]:
# Resolve presets and Colab guards (papermill-safe).
import os

# gene-only vs full-input presets
if GENE_ONLY:
    if not MODEL_ID:
        MODEL_ID = "Ha-ya/QWEN3-1.7B-EIGHT-ORGANS-GENE-ONLY-EXTENDED-1K"
    if not FILENAME_PATTERN:
        FILENAME_PATTERN = "{organ}_celldata_part_{part}_gene_only.json"
else:
    if not MODEL_ID:
        MODEL_ID = "Ha-ya/QWEN3-1.7B-EIGHT-ORGANS-EXTENDED-1K"
    if not FILENAME_PATTERN:
        FILENAME_PATTERN = "{organ}_cell_data_part_{part}_sorted.json"

if not DATA_DIR:
    DATA_DIR = os.environ.get("SCAP_DATA_DIR", f"{PROJECT_DIR}/fine_tune_chunks")
if not OUTPUT_DIR:
    OUTPUT_DIR = f"{PROJECT_DIR}/explain_outputs"

os.environ["PJRT_DEVICE"] = "TPU"
os.environ["XLA_USE_BF16"] = "1"

# Drive is mounted by the frontend; mount here only if explicitly requested.
if MOUNT_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped:", repr(exc))

# Optional HF auth from the environment (never hardcoded; never userdata in this child kernel).
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("HF login OK (token from environment).")
    except Exception as exc:
        print("HF login skipped:", repr(exc))

print("GENE_ONLY:", GENE_ONLY, "| MODEL_ID:", MODEL_ID)
print("DATA_DIR:", DATA_DIR, "| FILENAME_PATTERN:", FILENAME_PATTERN)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Runtime setup, dependencies, and project data

In [ ]:
# Runtime setup: Colab TPU environment, dependencies, and project data.
import os
import shutil
import subprocess
import sys

os.environ['PJRT_DEVICE'] = 'TPU'
os.environ['XLA_USE_BF16'] = '1'
os.environ.setdefault('PT_XLA_DEBUG', '0')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')


PYTORCH_VERSION = '2.6.0'
TARGET_DIR = PROJECT_DIR


def pip_install(args):
    print('pip', ' '.join(args))
    subprocess.check_call([sys.executable, '-m', 'pip', *args])


def has_json_data(path):
    return os.path.isdir(path) and any(name.endswith('.json') for name in os.listdir(path))


try:
    import torch
    import torch_xla
    print('torch:', torch.__version__)
    print('torch_xla:', torch_xla.__version__)
except Exception as exc:
    print('Installing torch/torch_xla because import failed:', repr(exc))
    pip_install([
        '-q', 'install',
        f'torch=={PYTORCH_VERSION}',
        f'torch_xla[tpu]=={PYTORCH_VERSION}',
        '-f', 'https://storage.googleapis.com/libtpu-releases/index.html',
    ])

pip_install([
    '-q', 'install', '-U',
    'transformers>=4.45.0',
    'datasets>=2.19.0',
    'scikit-learn>=1.3.0',
    'peft>=0.12.0',
    'evaluate>=0.4.2',
    'wordcloud',
])

# Match the original notebook behavior: start fresh in /content, then clone the working repo.
if not has_json_data(DATA_DIR):
    if not os.path.exists(TARGET_DIR):
        subprocess.check_call(['git', 'clone', REPO_URL, TARGET_DIR])
    if not os.environ.get('SCAP_DATA_DIR'):
        DATA_DIR = f'{TARGET_DIR}/fine_tune_chunks'
else:
    print('Using existing DATA_DIR:', DATA_DIR)

if not has_json_data(DATA_DIR):
    raise FileNotFoundError(
        f'No JSON files found in DATA_DIR: {DATA_DIR}. '
        'Set DATA_DIR or os.environ["SCAP_DATA_DIR"] to the folder containing the *_sorted.json files.'
    )

import gc
import glob
import inspect
import json
import pickle
import random
import time
from collections import Counter, defaultdict

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
from datasets import Dataset, DatasetDict, load_dataset
from peft import LoraConfig, get_peft_model
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from wordcloud import WordCloud

xr.use_spmd()
if not hasattr(torch, 'xla'):
    torch.xla = torch_xla

try:
    import torch.utils.checkpoint
    import torch_xla.utils.checkpoint as xla_ckpt
    torch.utils.checkpoint.checkpoint = xla_ckpt.checkpoint
except Exception as exc:
    print('Checkpoint patch skipped:', repr(exc))

%matplotlib inline
matplotlib.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

print('PJRT device:', os.environ.get('PJRT_DEVICE'))
print('Global runtime device count:', xr.global_runtime_device_count())
print('XLA device:', torch_xla.device())
print('DATA_DIR:', DATA_DIR)
print('Setup complete.')


## 3. Configuration (labels, dtype, save paths)

In [ ]:
# DTYPE string -> torch dtype; labels, seed, and save paths.
_DTYPE_MAP = {"bfloat16": torch.bfloat16, "float16": torch.float16, "float32": torch.float32}
DTYPE = _DTYPE_MAP.get(str(DTYPE).lower().replace("torch.", ""), torch.bfloat16)

LABEL2ID = {'1m': 0, '3m': 1, '18m': 2, '24m': 3, '30m': 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

RESULTS_DIR = OUTPUT_DIR
RESULTS_PKL = os.path.join(RESULTS_DIR, 'qwen3_1_7b_gradient_importance_results.pkl')
RESULTS_JSON = os.path.join(RESULTS_DIR, 'qwen3_1_7b_gradient_importance_results.json')

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)
print('MODEL_ID:', MODEL_ID, '| DTYPE:', DTYPE, '| GENE_ONLY:', GENE_ONLY)
print('Results dir:', RESULTS_DIR)


## 4. Load data

In [ ]:
import glob

print('Files found:', len(glob.glob(f'{DATA_DIR}/*.json')))
print('Example files:', sorted(glob.glob(f'{DATA_DIR}/*.json'))[:5])

ORGAN_STEM = {
    'bladder': 'bladder', 'brain': 'brain', 'bone': 'bone-marrow',
    'limb': 'limb-muscle', 'kidney': 'kidney', 'liver': 'liver',
    'lung': 'lung', 'heart': 'heart',
}

def load_json_dataset(files):
    try:
        return load_dataset('json', data_files=files, split='train')
    except Exception as exc:
        print('load_dataset(json) failed; falling back to manual JSON load. Error:', exc)
        rows = []
        for fp in files:
            with open(fp, 'r') as f:
                rows.extend(json.load(f))
        return Dataset.from_list(rows)

def organ_files(organ_name, parts=ALL_PARTS):
    stem = ORGAN_STEM.get(organ_name, organ_name)
    return [os.path.join(DATA_DIR, FILENAME_PATTERN.format(organ=stem, part=i)) for i in parts]

organ_file_map = {organ: organ_files(organ) for organ in ORGAN_KEYS}

# Fail loud if the selected pattern matches nothing (avoid silently feeding wrong data).
_expected = [fp for files in organ_file_map.values() for fp in files]
_missing = [fp for fp in _expected if not os.path.exists(fp)]
if len(_missing) == len(_expected):
    raise FileNotFoundError(
        f"No data files matched FILENAME_PATTERN={FILENAME_PATTERN!r} in DATA_DIR={DATA_DIR!r}. "
        f"For GENE_ONLY={GENE_ONLY}: set GENE_ONLY=False to use the committed *_sorted.json files, "
        f"or point DATA_DIR/SCAP_DATA_DIR at the gene-only data."
    )
if _missing:
    print(f"WARNING: {len(_missing)}/{len(_expected)} expected file(s) not found; loading the rest.")

organ_datasets = {organ: load_json_dataset([fp for fp in files if os.path.exists(fp)])
                  for organ, files in organ_file_map.items()}
all_files = [fp for fp in _expected if os.path.exists(fp)]
eval_ds = load_json_dataset(all_files)

raw = DatasetDict({'validation': eval_ds, **organ_datasets})
print(raw)
print('Columns:', raw['validation'].column_names)
print('Example output label:', raw['validation'][0].get('output'))
print('\nPer-organ sizes:')
for split in ORGAN_KEYS:
    print(f'  {split}: {len(raw[split])} samples')
print(f"  TOTAL validation: {len(raw['validation'])} samples")


## 5. Prepare text, tokenize, and load the model

In [ ]:
def format_text(example):
    instruction = (example.get('instruction') or '').strip()
    inp = (example.get('input') or '').strip()

    prefix = instruction + '\n\n'
    suffix = '\n\nAnswer with exactly one label from {1m, 3m, 18m, 24m, 30m}.'
    text = prefix + inp + suffix

    y = (example.get('output') or '').strip()
    if y not in LABEL2ID:
        raise ValueError(f'Unexpected label: {y}')

    return {
        'text': text,
        'label': LABEL2ID[y],
        'inp_start_char': len(prefix),
        'inp_end_char': len(prefix) + len(inp),
    }


raw = raw.map(format_text, remove_columns=raw['validation'].column_names)
print(raw['validation'][0])


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
if tokenizer.bos_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.bos_token = tokenizer.eos_token

print('Tokenizer pad_token:', tokenizer.pad_token)
print('Tokenizer vocab size:', tokenizer.vocab_size)


def tok_fn(batch):
    enc = tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
        return_offsets_mapping=True,
    )

    inp_token_masks = []
    for i in range(len(batch['text'])):
        start = batch['inp_start_char'][i]
        end = batch['inp_end_char'][i]
        mask = [1 if tok_start < end and tok_end > start else 0
                for tok_start, tok_end in enc['offset_mapping'][i]]
        inp_token_masks.append(mask)

    enc['inp_token_mask'] = inp_token_masks
    enc['labels'] = batch['label']
    del enc['offset_mapping']
    return enc


tok = raw.map(tok_fn, batched=True, remove_columns=raw['validation'].column_names)
print('Tokenized splits:', list(tok.keys()))
for split in tok.keys():
    print(f'  {split}: {len(tok[split])} samples')
print('tok ready.')


In [ ]:
def guess_lora_targets(model: torch.nn.Module):
    common = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    present = set()
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            present.update(k for k in common if name.endswith(k))
    return sorted(present or {'q_proj', 'v_proj'})


model_kwargs = {
    'num_labels': len(LABEL2ID),
    'id2label': ID2LABEL,
    'label2id': LABEL2ID,
    'trust_remote_code': True,
}
sig = inspect.signature(AutoModelForSequenceClassification.from_pretrained)
model_kwargs['dtype' if 'dtype' in sig.parameters else 'torch_dtype'] = DTYPE

model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, **model_kwargs)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
if getattr(model.config, 'bos_token_id', None) is None and tokenizer.bos_token_id is not None:
    model.config.bos_token_id = tokenizer.bos_token_id
if hasattr(model, 'generation_config') and getattr(model.generation_config, 'bos_token_id', None) is None:
    model.generation_config.bos_token_id = tokenizer.bos_token_id

print('Model loaded:', MODEL_ID, '| num_labels:', model.config.num_labels)

lora_targets = sorted(set(guess_lora_targets(model) + ['embed_tokens']))
print('LoRA target_modules:', lora_targets)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type='SEQ_CLS',
    target_modules=lora_targets,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.config.use_cache = False

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=128)
device = torch_xla.device()
model = model.to(device)
torch_xla.sync()
print(f'Model on device: {device}')


## 6. Evaluation and gradient-importance helpers

In [ ]:
def _get_logits(predictions):
    # Some models return tuple/list; keep logits only
    if isinstance(predictions, (tuple, list)):
        return predictions[0]
    return predictions

labels_sorted = sorted(list(ID2LABEL.keys()))
label_names = [ID2LABEL[i] for i in labels_sorted]

# ------------------------------
# Subset -> validation index mapping helpers
# ------------------------------

_ID_CANDIDATES = ["id", "idx", "guid", "example_id", "uid", "record_id"]

def _try_get_indices_from__indices(subset_ds):
    """
    If subset_ds was produced from a parent dataset via .select/.filter,
    HF Datasets often stores the parent indices in subset_ds._indices.
    Returns np.ndarray[int64] or None.
    """
    idx_tbl = getattr(subset_ds, "_indices", None)
    if idx_tbl is None:
        return None

    try:
        # datasets.table.Table supports .column_names and .column(i)
        col_names = getattr(idx_tbl, "column_names", None)
        if col_names and len(col_names) > 0:
            # Prefer a column literally named "indices" if present, else first column
            if "indices" in col_names:
                col = idx_tbl.column(col_names.index("indices"))
            else:
                col = idx_tbl.column(0)
            idx_list = col.to_pylist()
            return np.asarray(idx_list, dtype=np.int64)
    except Exception:
        pass

    # Some versions may store as dict-like
    try:
        if hasattr(idx_tbl, "to_pydict"):
            d = idx_tbl.to_pydict()
            if "indices" in d:
                return np.asarray(d["indices"], dtype=np.int64)
            # fallback: first key
            if len(d) > 0:
                k0 = list(d.keys())[0]
                return np.asarray(d[k0], dtype=np.int64)
    except Exception:
        pass

    return None

def _pick_id_column(ds):
    cols = set(getattr(ds, "column_names", []))
    for c in _ID_CANDIDATES:
        if c in cols:
            return c
    return None

def _try_get_indices_from_id_column(subset_ds, parent_ds):
    """
    If both parent and subset share a stable ID column, build mapping parent_id -> index.
    Returns np.ndarray[int64] or None.
    """
    parent_id_col = _pick_id_column(parent_ds)
    subset_id_col = _pick_id_column(subset_ds)
    if parent_id_col is None or subset_id_col is None or parent_id_col != subset_id_col:
        return None

    id_col = parent_id_col
    try:
        parent_ids = parent_ds[id_col]
        subset_ids = subset_ds[id_col]
    except Exception:
        return None

    # Build parent map (first occurrence wins)
    id2i = {}
    for i, v in enumerate(parent_ids):
        if v not in id2i:
            id2i[v] = i

    idx = []
    missing = 0
    for v in subset_ids:
        if v in id2i:
            idx.append(id2i[v])
        else:
            missing += 1

    if missing > 0:
        # If anything is missing, mapping is unreliable
        return None

    return np.asarray(idx, dtype=np.int64)

def get_subset_indices_in_validation(subset_ds, validation_ds):
    """
    Returns indices into validation_ds corresponding to subset_ds, or None if cannot map.
    """
    idx = _try_get_indices_from__indices(subset_ds)
    if idx is not None:
        return idx

    idx = _try_get_indices_from_id_column(subset_ds, validation_ds)
    if idx is not None:
        return idx

    return None

# ------------------------------
# Core evaluation from arrays (no extra inference)
# ------------------------------

def evaluate_from_arrays(split_name: str, y_true: np.ndarray, logits: np.ndarray, metrics: dict = None):
    """
    Compute confusion matrix, per-label recall (row-wise acc), and classification report
    from already-available y_true and logits.
    """
    y_true = np.asarray(y_true).reshape(-1)
    logits = np.asarray(logits)
    y_pred = np.argmax(logits, axis=-1).reshape(-1)

    cm = confusion_matrix(y_true, y_pred, labels=labels_sorted)

    # Per-label row-wise accuracy == recall from CM
    per_label_acc = {}
    for row_i, lab_id in enumerate(labels_sorted):
        denom = cm[row_i, :].sum()
        per_label_acc[ID2LABEL[lab_id]] = float(cm[row_i, row_i] / denom) if denom > 0 else float("nan")

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=labels_sorted,
        target_names=label_names,
        digits=4,
        output_dict=True,
        zero_division=0,
    )

    # Derived summary metrics (consistent across splits)
    derived = {
        "accuracy": float(report_dict.get("accuracy", float("nan"))),
        "macro_precision": float(report_dict.get("macro avg", {}).get("precision", float("nan"))),
        "macro_recall": float(report_dict.get("macro avg", {}).get("recall", float("nan"))),
        "macro_f1": float(report_dict.get("macro avg", {}).get("f1-score", float("nan"))),
        "weighted_f1": float(report_dict.get("weighted avg", {}).get("f1-score", float("nan"))),
    }

    # Store metrics with a consistent prefix for plotting helpers
    overall_metrics = {}
    if metrics:
        overall_metrics.update(metrics)

    # Ensure we always have an "*accuracy" key for plotting (even for sub-splits)
    overall_metrics[f"final_{split_name}_accuracy"] = derived["accuracy"]
    overall_metrics[f"final_{split_name}_macro_f1"] = derived["macro_f1"]

    return {
        "split": split_name,
        "num_samples": int(len(y_true)),
        "overall_metrics": overall_metrics,
        "derived_metrics": derived,
        "per_label_accuracy_rowwise": per_label_acc,
        "confusion_matrix": {"labels": label_names, "matrix": cm.tolist()},
        "classification_report": report_dict,
    }

def _extract_overall_acc(metrics: dict, report_dict: dict):
    # Prefer trainer metric if present; fallback to classification_report accuracy.
    acc_key = next((k for k in metrics.keys() if k.endswith("accuracy")), None)
    if acc_key is not None and isinstance(metrics.get(acc_key, None), (int, float)):
        return float(metrics[acc_key])
    if "accuracy" in report_dict and isinstance(report_dict["accuracy"], (int, float)):
        return float(report_dict["accuracy"])
    return float("nan")

# ------------------------------
# Plotting (unchanged behavior, now used for each split)
# ------------------------------

def plot_accuracy_f1_recall(split_result: dict, out_dir: str):
    """
    Figure 1:
      - Per-class Recall (row-wise accuracy) + Per-class F1 (grouped bars)
      - Overall accuracy (horizontal line)
      - Macro Recall / Macro F1 (text)
    """
    labels = split_result["confusion_matrix"]["labels"]
    report = split_result["classification_report"]
    metrics = split_result["overall_metrics"]

    recalls = []
    f1s = []
    supports = []
    for lab in labels:
        r = split_result["per_label_accuracy_rowwise"].get(lab, float("nan"))
        recalls.append(r)

        if lab in report:
            f1s.append(float(report[lab].get("f1-score", 0.0)))
            supports.append(int(report[lab].get("support", 0)))
        else:
            f1s.append(0.0)
            supports.append(0)

    recalls_plot = [0.0 if (r is None or not np.isfinite(r)) else float(r) for r in recalls]

    overall_acc = _extract_overall_acc(metrics, report)
    macro_recall = float(report.get("macro avg", {}).get("recall", float("nan")))
    macro_f1 = float(report.get("macro avg", {}).get("f1-score", float("nan")))

    x = np.arange(len(labels))
    width = 0.40

    plt.figure(figsize=(max(10, 0.7 * len(labels)), 6))
    plt.bar(x - width/2, recalls_plot, width, label="Per-class Recall (row-wise acc)")
    plt.bar(x + width/2, f1s, width, label="Per-class F1")

    if np.isfinite(overall_acc):
        plt.axhline(overall_acc, linestyle="--", linewidth=2, label=f"Overall Accuracy = {overall_acc:.4f}")

    xticks = [f"{lab}\n(n={n})" for lab, n in zip(labels, supports)]
    plt.xticks(x, xticks, rotation=45, ha="right")
    plt.ylim(0, 1.05)
    plt.ylabel("Score")
    plt.title(f"[{split_result['split']}] Per-class Recall/F1 + Overall Accuracy")

    text_lines = []
    if np.isfinite(macro_recall):
        text_lines.append(f"Macro Recall: {macro_recall:.4f}")
    if np.isfinite(macro_f1):
        text_lines.append(f"Macro F1: {macro_f1:.4f}")
    if np.isfinite(overall_acc):
        text_lines.append(f"Overall Acc: {overall_acc:.4f}")
    if text_lines:
        plt.gca().text(
            1.01, 0.5,
            "\n".join(text_lines),
            transform=plt.gca().transAxes,
            va="center",
            fontsize=11,
            bbox=dict(boxstyle="round", alpha=0.2),
        )

    plt.legend()
    plt.tight_layout()

    os.makedirs(out_dir, exist_ok=True)
    fig_path = os.path.join(out_dir, f"final_{split_result['split']}_metrics.png")
    plt.savefig(fig_path, dpi=200)
    plt.show()
    print(f"Saved metrics figure to: {fig_path}")

def plot_confusion_matrix(split_result: dict, out_dir: str):
    """
    Figure 2:
      - Confusion matrix with row-normalized heatmap
      - Annotate each cell with: count + normalized value
    """
    labels = split_result["confusion_matrix"]["labels"]
    cm = np.array(split_result["confusion_matrix"]["matrix"], dtype=np.int64)

    row_sums = cm.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        cm_norm = np.divide(cm, row_sums, where=(row_sums != 0))
        cm_norm = np.nan_to_num(cm_norm)

    plt.figure(figsize=(max(8, 0.6 * len(labels)), max(6, 0.6 * len(labels))))
    im = plt.imshow(cm_norm)  # default colormap is fine
    plt.colorbar(im, fraction=0.046, pad=0.04)

    plt.xticks(np.arange(len(labels)), labels, rotation=45, ha="right")
    plt.yticks(np.arange(len(labels)), labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"[{split_result['split']}] Confusion Matrix (row-normalized colors)")

    for i in range(len(labels)):
        for j in range(len(labels)):
            plt.text(
                j, i,
                f"{cm[i, j]}\n{cm_norm[i, j]:.2f}",
                ha="center", va="center",
                fontsize=9
            )

    plt.tight_layout()

    os.makedirs(out_dir, exist_ok=True)
    fig_path = os.path.join(out_dir, f"final_{split_result['split']}_confusion_matrix.png")
    plt.savefig(fig_path, dpi=200)
    plt.show()
    print(f"Saved confusion matrix figure to: {fig_path}")


In [ ]:
# ============================================================
# BLOCK 1: Gradient importance analysis — batched, use_true_label=True
# ============================================================
# Key design decisions:
#   - GRAD_BATCH_SIZE = 8 (lower this if TPU memory is tight)
#   - use_true_label=True : attribute toward ground-truth class
#   - Batching: transformer Jacobian is block-diagonal across the batch, so
#     d(Σ_b logit[b,target[b]]) / d(embeds[b]) == d(logit[b,target[b]]) / d(embeds[b])
#   - torch_xla.sync() once per batch → ~batch_size× speedup vs per-sample
#   - output_hidden_states=False + explicit del of XLA tensors → no HBM leak
#   - model.zero_grad(set_to_none=True) AFTER backward frees ~3.4 GB param grads
# ============================================================

# ── Batch size ────────────────────────────────────────────────────────────────
GRAD_BATCH_SIZE = globals().get('GRAD_BATCH_SIZE', 8)
GRAD_TOPK = globals().get('GRAD_TOPK', 100)


# ---------- Token cleaning ----------
def _clean_token(tok_str: str) -> str:
    return tok_str.replace("▁", " ").replace("Ġ", " ").strip()


# ---------- Batched gradient importance (memory-safe) ----------
def compute_gradient_importance_batch(model, tokenizer, examples, device,
                                       topk=None, use_true_label=True):
    """
    Compute gradient×input saliency for a batch of examples in ONE forward+backward.

    Memory contract  (revised — aggressively frees CPU + XLA memory)
    ───────────────
    1. Raw embedding tensor (_raw) is separated and deleted before forward pass.
    2. After backward: model.zero_grad(set_to_none=True) frees ~3.4 GB of
       parameter gradients that backward() allocated.
    3. Saliency is computed in a single fused expression (no extra named tensors).
    4. embeds.grad is set to None, then embeds is deleted.
    5. torch_xla.sync() + gc.collect() at the end to force XLA buffer reclamation.
    """
    if topk is None:
        topk = globals().get('GRAD_TOPK', 100)

    B = len(examples)

    input_ids_list = [ex["input_ids"]     for ex in examples]
    attn_mask_list = [ex["attention_mask"] for ex in examples]
    inp_mask_list  = [ex["inp_token_mask"] for ex in examples]
    true_labels    = [int(ex["labels"]) if "labels" in ex else None for ex in examples]

    # ── Build XLA tensors ─────────────────────────────────────────────────────
    input_ids_t = torch.tensor(input_ids_list, dtype=torch.long).to(device)
    attn_mask_t = torch.tensor(attn_mask_list, dtype=torch.long).to(device)

    # Embedding lookup — separate raw tensor so we can free it immediately
    _raw = model.get_input_embeddings()(input_ids_t)
    del input_ids_t                                    # ← free: not needed after lookup

    embeds = _raw.detach().clone().requires_grad_(True)  # [B, L, H]
    del _raw                                           # ← free autograd-graph-carrying tensor

    # ── Forward pass ─────────────────────────────────────────────────────────
    out = model(
        inputs_embeds=embeds,
        attention_mask=attn_mask_t,
        output_attentions=False,
        output_hidden_states=False,
        return_dict=True,
    )
    del attn_mask_t

    logits = out.logits
    del out

    pred_classes = torch.argmax(logits, dim=-1).cpu().tolist()

    # ── Determine target class per sample ─────────────────────────────────────
    target_classes = []
    for b in range(B):
        if use_true_label and true_labels[b] is not None:
            target_classes.append(true_labels[b])
        else:
            target_classes.append(pred_classes[b])

    # ── Backward pass ─────────────────────────────────────────────────────────
    target_logit_sum = sum(logits[b, target_classes[b]] for b in range(B))
    del logits

    model.zero_grad(set_to_none=True)                  # free leftover grads from prev batch
    target_logit_sum.backward()
    del target_logit_sum

    # ── Saliency → CPU numpy in one shot, then nuke all XLA refs ─────────────
    with torch.no_grad():
        grad_scores_batch = (embeds.grad * embeds).float().norm(dim=-1).cpu().numpy()

    # Free ~3.4 GB of parameter gradients that backward() just created
    model.zero_grad(set_to_none=True)

    # Break the grad reference, then delete the leaf tensor
    embeds.grad = None
    del embeds

    # Flush XLA lazy graph + force Python GC so the XLA allocator reclaims now
    torch_xla.sync()
    gc.collect()

    # ── Build per-sample results (pure Python/NumPy from here) ────────────────
    results = []
    for b in range(B):
        attn_mask_np       = np.array(attn_mask_list[b], dtype=np.int64)
        inp_mask_np        = np.array(inp_mask_list[b],  dtype=np.int64)
        grad_scores        = grad_scores_batch[b]
        grad_scores_masked = grad_scores.copy()
        grad_scores_masked[attn_mask_np == 0] = -np.inf
        grad_scores_masked[inp_mask_np  == 0] = -np.inf

        token_strs = tokenizer.convert_ids_to_tokens(input_ids_list[b])
        token_strs = [_clean_token(t) for t in token_strs]

        SPECIAL = {"", "[PAD]", "<pad>", "<s>", "</s>", "<unk>", "<|endoftext|>"}
        valid_indices = [
            i for i, t in enumerate(token_strs)
            if t not in SPECIAL and attn_mask_np[i] == 1 and inp_mask_np[i] == 1
        ]

        if not valid_indices:
            results.append(([], [], pred_classes[b], true_labels[b]))
            continue

        valid_scores = grad_scores_masked[valid_indices]
        top_k_local  = min(topk, len(valid_indices))
        top_k_idx    = np.argsort(-valid_scores)[:top_k_local]

        top_tokens = [token_strs[valid_indices[i]] for i in top_k_idx]
        top_scores = valid_scores[top_k_idx].tolist()
        results.append((top_tokens, top_scores, pred_classes[b], true_labels[b]))

    return results


# ---------- Main analysis loop (memory-safe) ----------
def run_gradient_analysis(model, tokenizer, tok, organ_keys,
                           samples_per_organ=None, device=None,
                           batch_size=None, topk=None, use_true_label=True):
    """
    Run batched gradient importance analysis on all organs.

    Memory strategy
    ───────────────
    - compute_gradient_importance_batch now handles sync + gc internally
    - Extra gc.collect() every GC_EVERY_N_BATCHES as safety net
    """
    GC_EVERY_N_BATCHES = 1    # gc every batch — critical for preventing RAM drift

    if device is None:
        device = torch_xla.device()
    if batch_size is None:
        batch_size = globals().get('GRAD_BATCH_SIZE', 8)
    if topk is None:
        topk = globals().get('GRAD_TOPK', 100)
    model.eval()

    organ_freq    = defaultdict(Counter)
    organ_scores  = defaultdict(Counter)
    global_freq   = Counter()
    global_scores = Counter()

    samples_processed = defaultdict(int)
    total_time        = 0.0

    for organ in organ_keys:
        if organ not in tok:
            print(f"[WARN] '{organ}' not found in tok, skipping.")
            continue

        organ_ds  = tok[organ]
        n_samples = len(organ_ds)
        if n_samples == 0:
            print(f"[WARN] '{organ}' dataset is empty, skipping.")
            continue

        n_to_process = n_samples if samples_per_organ is None else min(samples_per_organ, n_samples)
        print(f"[{organ.upper()}] Processing {n_to_process}/{n_samples} samples "
              f"(batch_size={batch_size}, use_true_label={use_true_label})...")

        batch_num = 0
        idx = 0
        while idx < n_to_process:
            batch_end = min(idx + batch_size, n_to_process)
            examples  = [organ_ds[i] for i in range(idx, batch_end)]
            t_start = time.perf_counter()

            batch_results = compute_gradient_importance_batch(
                model, tokenizer, examples, device,
                topk=topk, use_true_label=use_true_label,
            )
            # NOTE: torch_xla.sync() + gc.collect() already called inside
            # compute_gradient_importance_batch — no need to sync again here.
            del examples   # ← free raw Python dicts

            elapsed     = time.perf_counter() - t_start
            total_time += elapsed

            # Aggregate
            for top_tokens, top_scores, pred_class, true_label in batch_results:
                for tok_str, score in zip(top_tokens, top_scores):
                    organ_freq[organ][tok_str]   += 1
                    organ_scores[organ][tok_str] += score
                    global_freq[tok_str]          += 1
                    global_scores[tok_str]        += score
                samples_processed[organ] += 1
            del batch_results

            idx       += batch_size
            batch_num += 1

            # Extra safety-net GC (lightweight if nothing to collect)
            if batch_num % GC_EVERY_N_BATCHES == 0:
                gc.collect()

            # Progress log every 50 samples or at end
            if idx % 50 < batch_size or idx >= n_to_process:
                total_so_far   = sum(samples_processed.values())
                avg_per_sample = total_time / total_so_far if total_so_far > 0 else 0.0
                print(f"    [{organ}] {min(idx, n_to_process)}/{n_to_process} done, "
                      f"avg {avg_per_sample:.2f}s/sample")

        # End-of-organ cleanup
        torch_xla.sync()
        gc.collect()
        print(f"  -> [{organ.upper()}] Done. Running total time: {total_time:.2f}s")

    # ── Avg-per-sample normalization (fair cross-organ comparison) ────────────
    organ_scores_avg = {}
    for _organ, _tok_scores in organ_scores.items():
        _n = samples_processed.get(_organ, 1) or 1
        organ_scores_avg[_organ] = Counter({_t: _s / _n for _t, _s in _tok_scores.items()})

    _global_n = sum(samples_processed.values()) or 1
    global_scores_avg = Counter({_t: _s / _global_n for _t, _s in global_scores.items()})

    organ_freq_avg = {}
    for _organ, _tok_freq in organ_freq.items():
        _n = samples_processed.get(_organ, 1) or 1
        organ_freq_avg[_organ] = Counter({_t: _c / _n for _t, _c in _tok_freq.items()})
    global_freq_avg = Counter({_t: _c / _global_n for _t, _c in global_freq.items()})

    return {
        "organ_freq":        dict(organ_freq),
        "organ_scores":      dict(organ_scores),
        "organ_scores_avg":  organ_scores_avg,
        "global_freq":       global_freq,
        "global_scores":     global_scores,
        "global_scores_avg": global_scores_avg,
        "organ_freq_avg":    organ_freq_avg,
        "global_freq_avg":   global_freq_avg,
        "samples_processed": dict(samples_processed),
        "total_time":        total_time,
    }

def visualize_results(results, organ_keys, out_dir=None, top_n=None, bar_color=None, wordcloud_cmap=None, figsize=None):
    """Print and plot global gradient-importance summaries without recomputing gradients."""
    if top_n is None:
        top_n = globals().get('PLOT_TOP_N', 100)
    if bar_color is None:
        bar_color = globals().get('PLOT_BAR_COLOR', 'steelblue')
    if wordcloud_cmap is None:
        wordcloud_cmap = globals().get('PLOT_WORDCLOUD_CMAP', 'plasma')
    if figsize is None:
        figsize = globals().get('PLOT_FIGSIZE', (16, 20))
    global_freq = Counter(results.get('global_freq', {}))
    global_scores = Counter(results.get('global_scores', {}))
    global_freq_avg = Counter(results.get('global_freq_avg', {}))

    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    print('\n' + '=' * 60)
    print('VISUALIZATION RESULTS')
    print('=' * 60)
    print(f"Samples processed: {results.get('samples_processed', {})}")
    print(f"Total time: {results.get('total_time', 0.0):.2f}s")

    top_global_freq = global_freq.most_common(top_n)
    print(f'\n--- Top {top_n} tokens by frequency (all organs) ---')
    for tok_str, count in top_global_freq:
        print(f"  '{tok_str}': {count}")
    if top_global_freq:
        tokens = [t for t, _ in top_global_freq]
        counts = [c for _, c in top_global_freq]
        plt.figure(figsize=figsize)
        plt.barh(range(len(tokens)), counts[::-1], color=bar_color)
        plt.yticks(range(len(tokens)), tokens[::-1], fontsize=10)
        plt.xlabel(f'Frequency in Top-{top_n}')
        plt.title(f'[ALL ORGANS] Top {top_n} Tokens by Frequency')
        plt.tight_layout()
        if out_dir:
            plt.savefig(os.path.join(out_dir, 'freq_global.png'), dpi=150)
        plt.show()

    top_global_scores = global_scores.most_common(top_n)
    print(f'\n--- Top {top_n} tokens by total score (all organs) ---')
    for tok_str, score in top_global_scores:
        print(f"  '{tok_str}': {score:.4f}")
    if top_global_scores:
        tokens = [t for t, _ in top_global_scores]
        scores = [s for _, s in top_global_scores]
        plt.figure(figsize=figsize)
        plt.barh(range(len(tokens)), scores[::-1], color=bar_color)
        plt.yticks(range(len(tokens)), tokens[::-1])
        plt.xlabel('Total Gradient Score')
        plt.title(f'[ALL ORGANS] Top {top_n} Tokens by Total Score')
        plt.tight_layout()
        if out_dir:
            plt.savefig(os.path.join(out_dir, 'score_global.png'), dpi=150)
        plt.show()

    top_global_freq_avg = global_freq_avg.most_common(top_n)
    print(f'\n--- Top {top_n} tokens by average frequency (global) ---')
    for tok_str, frac in top_global_freq_avg:
        print(f"  '{tok_str}': {frac:.4f}")
    if top_global_freq_avg:
        tokens = [t for t, _ in top_global_freq_avg]
        fracs = [f for _, f in top_global_freq_avg]
        plt.figure(figsize=figsize)
        plt.barh(range(len(tokens)), fracs[::-1])
        plt.yticks(range(len(tokens)), tokens[::-1])
        plt.xlabel(f'Average frequency in Top-{top_n}')
        plt.title(f'[ALL ORGANS] Top {top_n} Tokens by Average Frequency')
        plt.tight_layout()
        if out_dir:
            plt.savefig(os.path.join(out_dir, 'avg_freq_global.png'), dpi=150)
        plt.show()

    print('\n--- Word cloud (all organs, by total score) ---')
    wc_dict = {t: max(s, 1e-6) for t, s in top_global_scores if s > 0}
    if len(wc_dict) >= 5:
        wc = WordCloud(width=800, height=600, background_color='white', max_words=top_n, colormap=wordcloud_cmap)
        wc.generate_from_frequencies(wc_dict)
        plt.figure(figsize=(16, 12))
        plt.imshow(wc, interpolation='bilinear')
        plt.axis('off')
        plt.title(f'[ALL ORGANS] Word Cloud (Top {top_n} by Score)')
        plt.tight_layout()
        if out_dir:
            plt.savefig(os.path.join(out_dir, 'wordcloud_global.png'), dpi=150)
        plt.show()

    print('\n' + '=' * 60)
    print('VISUALIZATION COMPLETE')
    if out_dir:
        print(f'Plots saved to: {out_dir}')
    print('=' * 60)
    return top_global_scores

def analyze_organ_specificity(results, organ_keys, topk=100, out_dir=None):
    """
    Identify tokens universally important vs. organ-exclusive.
    Uses avg-per-sample scores for fair cross-organ comparison.
    """
    organ_scores_src = results.get("organ_scores_avg") or results.get("organ_scores", {})

    organ_topk_sets = {}
    for organ in organ_keys:
        if organ not in organ_scores_src:
            continue
        top_tokens = [t for t, _ in Counter(organ_scores_src[organ]).most_common(topk)]
        organ_topk_sets[organ] = set(top_tokens)

    if not organ_topk_sets:
        print("[WARN] No organ data found in results.")
        return {}

    token_organ_count = Counter()
    for organ, tok_set in organ_topk_sets.items():
        for t in tok_set:
            token_organ_count[t] += 1

    n_organs  = len(organ_topk_sets)
    universal = [t for t, c in token_organ_count.most_common() if c == n_organs]

    exclusive = {organ: [] for organ in organ_keys}
    for t, c in token_organ_count.items():
        if c == 1:
            for organ, tok_set in organ_topk_sets.items():
                if t in tok_set:
                    exclusive[organ].append(t)
                    break

    print("\n" + "="*60)
    print("ORGAN SPECIFICITY ANALYSIS")
    print("="*60)
    print(f"\nTokens in top-{topk} for ALL {n_organs} organs ({len(universal)} tokens):")
    for t in universal[:30]:
        print(f"  '{t}'")
    print("\nOrgan-exclusive tokens (unique to each organ's top-k):")
    for organ in organ_keys:
        excl = exclusive.get(organ, [])
        print(f"  [{organ.upper()}] {len(excl)} exclusive: {excl[:20]}")

    # Frequency-based specificity
    _ofsrc = results.get("organ_freq_avg") or results.get("organ_freq", {})
    _otfs  = {o: set(t for t, _ in Counter(_ofsrc.get(o, {})).most_common(topk)) for o in organ_keys}
    _tfc   = Counter()
    for _s in _otfs.values():
        for _t in _s:
            _tfc[_t] += 1
    _no = len([o for o in organ_keys if _otfs.get(o)])
    _uf = [t for t, c in _tfc.most_common() if c == _no]
    _ef = {o: [t for t, c in _tfc.items() if c == 1 and t in _otfs.get(o, set())] for o in organ_keys}
    print("\n--- Frequency-based specificity ---")
    print(f"Universal by freq ({len(_uf)} tokens): {_uf[:30]}")
    print("Exclusive by freq:")
    for o in organ_keys:
        print(f"  [{o.upper()}] {len(_ef.get(o,[]))}: {_ef.get(o,[])[:20]}")

    # Pairwise overlap heatmap
    organ_list = [o for o in organ_keys if o in organ_topk_sets]
    n = len(organ_list)
    if n > 1:
        overlap_matrix = np.zeros((n, n), dtype=int)
        for i in range(n):
            for j in range(n):
                overlap_matrix[i, j] = len(organ_topk_sets[organ_list[i]] & organ_topk_sets[organ_list[j]])

        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(overlap_matrix, cmap="YlOrRd")
        ax.set_xticks(range(n)); ax.set_yticks(range(n))
        ax.set_xticklabels(organ_list, rotation=45, ha="right")
        ax.set_yticklabels(organ_list)
        plt.colorbar(im, ax=ax, label=f"# shared tokens in top-{topk}")
        for i in range(n):
            for j in range(n):
                ax.text(j, i, str(overlap_matrix[i, j]), ha="center", va="center", fontsize=10)
        ax.set_title(f"Organ pairwise token overlap (top-{topk} by avg gradient score)")
        plt.tight_layout()
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
            fig_path = os.path.join(out_dir, "organ_token_overlap.png")
            plt.savefig(fig_path, dpi=150)
            print(f"Saved: {fig_path}")
        plt.show()

    return {
        "universal_tokens":  universal,
        "exclusive_tokens":  exclusive,
        "token_organ_count": dict(token_organ_count),
        "organ_topk_sets":   {k: list(v) for k, v in organ_topk_sets.items()},
    }


## 7. Run the gradient analysis

In [ ]:
required = ['tok', 'model', 'tokenizer', 'run_gradient_analysis']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'Missing variables: {missing}. Run setup through gradient utility cells first.')

print('=' * 60)
limit_msg = 'all samples' if SAMPLES_PER_ORGAN is None else f'{SAMPLES_PER_ORGAN} samples per organ'
print(f'GRADIENT RUN: {limit_msg}, batch_size={GRAD_BATCH_SIZE}, topk={GRAD_TOPK}, use_true_label={USE_TRUE_LABEL}')
print('=' * 60)

print('\nDataset sizes (all parts per organ):')
total_samples = 0
for organ in ORGAN_KEYS:
    if organ in tok:
        n = len(tok[organ])
        n_run = n if SAMPLES_PER_ORGAN is None else min(SAMPLES_PER_ORGAN, n)
        print(f'  {organ}: {n_run}/{n}')
        total_samples += n_run
print(f'  TOTAL TO RUN: {total_samples}')

print('\nStarting gradient analysis...')
full_results = run_gradient_analysis(
    model,
    tokenizer,
    tok,
    ORGAN_KEYS,
    samples_per_organ=SAMPLES_PER_ORGAN,
    device=torch_xla.device(),
    batch_size=GRAD_BATCH_SIZE,
    topk=GRAD_TOPK,
    use_true_label=USE_TRUE_LABEL,
)

print(f"\nGradient analysis done. Total time: {full_results['total_time']:.1f}s")
print(f"Samples processed: {full_results['samples_processed']}")

if SAVE_RESULTS:
    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(RESULTS_PKL, 'wb') as f:
        pickle.dump(full_results, f)
    results_to_save = {
        'organ_freq': {k: dict(v) for k, v in full_results['organ_freq'].items()},
        'organ_scores': {k: dict(v) for k, v in full_results['organ_scores'].items()},
        'organ_scores_avg': {k: dict(v) for k, v in full_results['organ_scores_avg'].items()},
        'global_freq': dict(full_results['global_freq']),
        'global_scores': dict(full_results['global_scores']),
        'global_scores_avg': dict(full_results['global_scores_avg']),
        'organ_freq_avg': {k: dict(v) for k, v in full_results.get('organ_freq_avg', {}).items()},
        'global_freq_avg': dict(full_results.get('global_freq_avg', {})),
        'samples_processed': full_results['samples_processed'],
        'total_time': full_results['total_time'],
        'settings': {
            'model_id': MODEL_ID,
            'max_length': MAX_LENGTH,
            'grad_topk': GRAD_TOPK,
            'grad_batch_size': GRAD_BATCH_SIZE,
            'samples_per_organ': SAMPLES_PER_ORGAN,
            'use_true_label': USE_TRUE_LABEL,
        },
    }
    with open(RESULTS_JSON, 'w') as f:
        json.dump(results_to_save, f, indent=2)
    print(f'Saved pickle: {RESULTS_PKL}')
    print(f'Saved JSON summary: {RESULTS_JSON}')
else:
    print('SAVE_RESULTS=False -> not writing pickle/JSON.')


In [ ]:
# Run this cell when you only want to change plots after a completed gradient run.
# It lets you restart the runtime and skip model loading/gradient computation if the pickle exists.
LOAD_SAVED_RESULTS = False

if LOAD_SAVED_RESULTS or 'full_results' not in globals():
    if not os.path.exists(RESULTS_PKL):
        raise FileNotFoundError(f'No saved results found at: {RESULTS_PKL}')
    with open(RESULTS_PKL, 'rb') as f:
        full_results = pickle.load(f)
    print(f'Loaded saved gradient results from: {RESULTS_PKL}')
else:
    print('Using full_results already in memory.')

print('Samples processed:', full_results.get('samples_processed', {}))
print(f"Gradient runtime stored in results: {full_results.get('total_time', 0.0):.1f}s")


## 8. Visualize and summarize

In [ ]:
if 'full_results' not in globals():
    raise RuntimeError('full_results is not loaded. Run the gradient cell or the reload cell first.')

print('Generating visualizations from saved/in-memory results...')
global_top_scores_full = visualize_results(
    full_results,
    ORGAN_KEYS,
    out_dir=PLOT_OUT_DIR,
    top_n=PLOT_TOP_N,
    bar_color=PLOT_BAR_COLOR,
    wordcloud_cmap=PLOT_WORDCLOUD_CMAP,
    figsize=PLOT_FIGSIZE,
)

print('\nRunning organ specificity analysis...')
specificity_results = analyze_organ_specificity(full_results, ORGAN_KEYS, topk=PLOT_TOP_N, out_dir=PLOT_OUT_DIR)

print('\n' + '=' * 70)
print(f'SUMMARY: TOP-{PLOT_TOP_N} TOKENS PER ORGAN (by average gradient score)')
print('=' * 70)
for organ in ORGAN_KEYS:
    src = full_results.get('organ_scores_avg', {})
    if organ not in src:
        continue
    top_tokens = Counter(src[organ]).most_common(PLOT_TOP_N)
    print(f'\n[{organ.upper()}]')
    hdr = f"  {'Rank':<5} {'Token':<30} {'Avg Grad Score':>16}  {'Freq (total)':>12}"
    print(hdr)
    print('  ' + '-' * (len(hdr) - 2))
    freq_src = full_results.get('organ_freq', {}).get(organ, {})
    for rank, (token, score) in enumerate(top_tokens, 1):
        freq = freq_src.get(token, 0)
        print(f'  {rank:<5} {token:<30} {score:>16.6f}  {freq:>12}')

print('\n' + '=' * 70)
print(f'SUMMARY: TOP-{PLOT_TOP_N} TOKENS GLOBAL (by average gradient score, all organs)')
print('=' * 70)
global_avg_top = Counter(full_results.get('global_scores_avg', {})).most_common(PLOT_TOP_N)
hdr = f"  {'Rank':<5} {'Token':<30} {'Avg Grad Score':>16}  {'Total Freq':>10}  {'Organs':>6}"
print(hdr)
print('  ' + '-' * (len(hdr) - 2))
token_organ_count = specificity_results.get('token_organ_count', {})
global_freq = full_results.get('global_freq', {})
for rank, (token, score) in enumerate(global_avg_top, 1):
    freq = global_freq.get(token, 0)
    n_organs = token_organ_count.get(token, '?')
    print(f'  {rank:<5} {token:<30} {score:>16.6f}  {freq:>10}  {n_organs:>6}')

print('\n' + '=' * 70)
print(f'SUMMARY: UNIVERSAL TOKENS (in top-{PLOT_TOP_N} for all organs)')
print('=' * 70)
universal = specificity_results.get('universal_tokens', [])
print(f'  Total: {len(universal)}')
for i, token in enumerate(universal, 1):
    print(f"  {i:>3}. '{token}'")

print('\n' + '=' * 70)
print(f"SUMMARY: ORGAN-EXCLUSIVE TOKENS (unique to each organ's top-{PLOT_TOP_N})")
print('=' * 70)
exclusive = specificity_results.get('exclusive_tokens', {})
for organ in ORGAN_KEYS:
    excl = exclusive.get(organ, [])
    print(f'\n  [{organ.upper()}] {len(excl)} exclusive tokens:')
    for i, token in enumerate(excl, 1):
        score = full_results.get('organ_scores_avg', {}).get(organ, {}).get(token, 0)
        print(f"    {i:>3}. '{token}'  (avg score: {score:.6f})")

if 'torch_xla' in globals():
    torch_xla.sync()
print('\n' + '=' * 70)
print('PLOTTING + ORGAN SPECIFICITY COMPLETE')
print('=' * 70)


In [ ]:
# ============================================================
# TOP-2048 GLOBAL AVERAGE GRADIENT SCORE TOKENS
# Ranked list of tokens by avg gradient importance across all organs.
# ============================================================
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

_gsa = full_results.get('global_scores_avg', {})
if not _gsa:
    print('[WARN] global_scores_avg empty — run Block 2 first.')
else:
    top2048 = Counter(_gsa).most_common(2048)
    n_unique = len(_gsa)

    # ── Text table (all 2048) ─────────────────────────────────────────────────
    print('=' * 70)
    print('TOP-2048 TOKENS BY AVERAGE GRADIENT IMPORTANCE (ALL ORGANS)')
    print('=' * 70)
    print(f'Total unique tokens tracked: {n_unique}  |  Reporting top {len(top2048)}\n')
    hdr = '{:<6} {:<35} {:>18}'.format('Rank', 'Token', 'Avg Grad Score')
    print(hdr)
    print('-' * len(hdr))
    for rank, (token, score) in enumerate(top2048, 1):
        print(f'{rank:<6} {token:<35} {score:>18.6f}')
    print('\n' + '=' * 70)

    # ── Bar chart: top-128 by avg gradient score ──────────────────────────────
    top100 = top2048[:100]
    tokens_100  = [t for t, _ in top100]
    scores_100  = [s for _, s in top100]
    # reverse for horizontal bar (highest at top)
    tokens_100r = tokens_100[::-1]
    scores_100r = scores_100[::-1]

    fig, ax = plt.subplots(figsize=(14, 40))
    bars = ax.barh(range(len(tokens_100r)), scores_100r, color='steelblue')
    ax.set_yticks(range(len(tokens_100r)))
    ax.set_yticklabels(tokens_100r, fontsize=9)
    ax.set_xlabel('Avg Gradient Score (all organs)', fontsize=11)
    ax.set_title('Top-100 Tokens — Avg Gradient Importance (All Organs)', fontsize=13)
    # Annotate each bar with its rank
    for i, (bar, score) in enumerate(zip(bars, scores_100r)):
        rank = len(top100) - i
        ax.text(bar.get_width() * 1.005, bar.get_y() + bar.get_height() / 2,
                f'#{rank}  {score:.4f}', va='center', fontsize=7.5)
    plt.tight_layout()
    plt.show()

    # ── Score distribution (log-scale histogram) ─────────────────────────────
    all_scores = [s for _, s in top2048]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(all_scores, bins=80, color='teal', edgecolor='white', linewidth=0.4)
    ax.set_yscale('log')
    ax.set_xlabel('Avg Gradient Score', fontsize=11)
    ax.set_ylabel('Count (log scale)', fontsize=11)
    ax.set_title('Distribution of Avg Gradient Scores — Top-2048 Tokens', fontsize=12)
    plt.tight_layout()
    plt.show()

    # ── Cumulative score coverage ─────────────────────────────────────────────
    total_score = sum(all_scores)
    cum_frac = np.cumsum(all_scores) / total_score
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, len(cum_frac) + 1), cum_frac * 100, color='darkorange', linewidth=1.5)
    for pct, color in [(50, 'blue'), (80, 'green'), (95, 'red')]:
        idx = np.searchsorted(cum_frac, pct / 100)
        ax.axvline(idx + 1, linestyle='--', color=color, linewidth=1,
                   label=f'{pct}% coverage @ rank {idx+1}')
    ax.set_xlabel('Token rank (by avg score)', fontsize=11)
    ax.set_ylabel('Cumulative score coverage (%)', fontsize=11)
    ax.set_title('Cumulative Score Coverage — Top-2048 Tokens', fontsize=12)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f'\nScore statistics:')
    print(f'  Top-1  token: {top2048[0][0]!r:35s}  score={top2048[0][1]:.6f}')
    print(f'  Top-10 cumulative fraction: {cum_frac[9]*100:.1f}%')
    print(f'  Top-100 cumulative fraction: {cum_frac[99]*100:.1f}%')
    if len(cum_frac) >= 2048:
        print(f'  Top-2048 cumulative fraction: {cum_frac[2047]*100:.1f}%')


In [ ]:
if SAVE_RESULTS:
    if 'full_results' not in globals():
        raise RuntimeError('full_results is not loaded.')
    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(RESULTS_PKL, 'wb') as f:
        pickle.dump(full_results, f)
    print(f'Re-saved pickle results to: {RESULTS_PKL}')
else:
    print('SAVE_RESULTS=False -> skipping manual re-save.')


## 9. Optional runtime cleanup

In [ ]:
# Optional Colab cleanup. Disabled by default; under papermill this would kill the kernel.
if RELEASE_RUNTIME:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print("runtime.unassign skipped:", repr(exc))
else:
    print("RELEASE_RUNTIME=False -> keeping runtime alive.")
